In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🌳 Branching Strategies & Git Workflows Guide

**Effective source control patterns for team collaboration and safe deployments**

This guide covers:
- Git branching models (Git Flow, GitHub Flow, Trunk-Based)
- Feature branch workflows
- Release management branches
- Hotfix procedures
- Pull request best practices
- Merge strategies and conflicts
- Commit hygiene and history
- Tag management for releases
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Git Flow Model

```
         ┌─── feature/user-auth ──┐
         │                         ↓
    ┌────┴──────────────────────────────┐
    │         develop branch             │
    │  (integration branch)              │
    │                                    │
    └────┬───────────────────┬──────────┘
         │                   │
    ┌────▼─┐                 ┌──▼───────────────┐
release/1.0   hotfix/security-patch
     │                        │
     └────────────┬───────────┘
                  │
              ┌───▼────┐
              │ main   │ (production)
              │ v1.0   │
              └────────┘
```

### Git Flow Commands
```bash
# Start a feature
git flow feature start user-auth
# → Creates branch: feature/user-auth

# Finish feature (auto merge to develop)
git flow feature finish user-auth

# Start release
git flow release start 1.0
# → Creates branch: release/1.0

# Finish release (merge to main and develop)
git flow release finish 1.0

# Hotfix production issue
git flow hotfix start security-patch
git flow hotfix finish security-patch
```

### Workflow Best Practices
```bash
# Keep develop stable
# Only merge tested, reviewed PRs

# Before merging to main:
1. Create release branch (release/1.0)
2. Test thoroughly
3. Fix bugs in release branch
4. Merge to main with version tag
5. Merge back to develop

# For urgent production fixes:
1. Create hotfix from main (hotfix/security-patch)
2. Fix and test
3. Merge to main AND develop
4. Tag version
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. GitHub Flow (Simpler)

```
main branch (always deployable)
    ↑         ↑         ↑
    │         │         │
    └─ PR1   └─ PR2    └─ PR3
    └── feature/login
         └── feature/profile
              └── feature/search
```

### GitHub Flow Steps
```bash
# 1. Create feature branch from main
git checkout -b feature/login
git push origin feature/login

# 2. Make commits (small, focused)
git commit -m "Add login form"
git push

# 3. Create Pull Request
# (via GitHub UI)

# 4. Get review and feedback
# Make changes based on review

# 5. Merge to main (squash commits)
git checkout main
git pull
git merge --squash feature/login
git push

# 6. Delete feature branch
git branch -d feature/login
git push origin --delete feature/login

# 7. Deploy from main
```

### Best Practices
- Main branch always ready to deploy
- Each PR is one feature/fix
- Squash commits for clean history
- Delete merged branches
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 3. Trunk-Based Development

```
                trunk (main)
   ┌────────────────────────────────┐
   │ Every commit should pass tests │
   │ Deploy frequently (daily)      │
   └────────────────────────────────┘
           │    │    │    │
      ┌────┘    │    │    └─────┐
      │    ┌────┘    └────┐     │
  dev1 dev2 dev3         dev4  dev5
(short-lived feature branches, 1-2 days max)
```

### Trunk-Based Workflow
```bash
# 1. Keep main always deployable
# 2. Short-lived feature branches (1-2 days)
git checkout -b feature/auth-v2
# ... make changes
git commit -m "Add 2FA support"

# 3. Quick code review and merge
git push origin feature/auth-v2
# Create PR, get approval

git checkout main
git merge feature/auth-v2
git push

# 4. Delete feature branch immediately
git branch -d feature/auth-v2

# 5. Deploy continuously
# Feature flags hide incomplete work
```

### Feature Flags for Incomplete Work
```python
@app.post("/api/login")
async def login(username: str, password: str):
    if is_feature_enabled("2fa", user_id=request.user_id):
        # New 2FA flow
        return await login_with_2fa(username, password)
    else:
        # Old flow
        return await login_classic(username, password)
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 4. Pull Request Best Practices

### PR Description Template
```markdown
## Description
Brief description of what this PR does.

## Type of Change
- [ ] Bug fix
- [ ] New feature
- [ ] Breaking change
- [ ] Documentation update

## Related Issues
Closes #123

## Testing
- [ ] Unit tests added
- [ ] Integration tests added
- [ ] Tested locally with commands:
  ```
  make test
  make lint
  ```

## Checklist
- [ ] My code follows the style guidelines
- [ ] I have performed a self-review
- [ ] Comments added for complex logic
- [ ] Documentation updated
- [ ] No breaking changes
- [ ] Tests pass
```

### Commit Message Format
```
type(scope): brief description

Longer explanation if needed. Explain WHY, not WHAT.

Fixes #123
```

Types: `feat`, `fix`, `docs`, `style`, `refactor`, `test`, `chore`

Examples:
```
feat(auth): add 2FA support
Implement time-based OTP for enhanced security. Uses pyotp library.
Fixes #456

fix(chat): handle connection timeouts
Add exponential backoff for reconnection attempts.
Fixes #789
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 5. Handling Merge Conflicts

### Prevent Conflicts
```bash
# Keep feature branches short-lived
# Frequently pull latest main
git pull origin main

# Communicate with team about changes
```

### Resolve Conflicts
```bash
# When merging feature into main
git checkout feature/auth
git fetch origin
git merge origin/main

# Git marks conflicts with <<<<<<< ======= >>>>>>>
# Edit files to resolve

# After resolving
git add .
git commit -m "Merge main into feature/auth"
git push

# Create PR for merge
```

### Conflict-Free Merge Strategies
```bash
# Rebase instead of merge (linear history)
git fetch origin
git rebase origin/main
git push -f  # Force push to feature branch

# Then when merging to main
git merge --ff-only feature/auth
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 6. Release Management

### Semantic Versioning
```
MAJOR.MINOR.PATCH
1.2.3

- MAJOR: Breaking changes (increment when)
- MINOR: Backward-compatible features (v1.2 → v1.3)
- PATCH: Bug fixes (v1.2.3 → v1.2.4)

Tags: v1.2.3
```

### Release Checklist
```bash
# 1. Create release branch
git checkout -b release/1.2.0

# 2. Update version numbers
# - package.json
# - pyproject.toml
# - __init__.py

# 3. Update CHANGELOG.md
```markdown
## [1.2.0] - 2026-07-05

### Added
- 2FA authentication support
- WebSocket support for real-time chat

### Fixed
- Memory leak in connection pool
- Race condition in cache invalidation

### Changed
- Improved performance 50%
```

# 4. Test release
make test
make lint

# 5. Create annotated tag
git tag -a v1.2.0 -m "Release 1.2.0"

# 6. Push release
git push origin release/1.2.0
git push origin v1.2.0

# 7. Merge to main
git checkout main
git merge --no-ff release/1.2.0
git push

# 8. Merge back to develop
git checkout develop
git merge --no-ff release/1.2.0
git push

# 9. Delete release branch
git branch -d release/1.2.0
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 7. Branch Protection Rules

### GitHub Branch Protection
```
Settings → Branches → Branch Protection Rules

Rules for main:
✓ Require pull request reviews (at least 1)
✓ Require status checks to pass (CI/CD)
✓ Require branches to be up to date
✓ Include administrators
✓ Restrict who can push to matching branches
✓ Require signed commits
```

### Automated Status Checks
```yaml
# .github/workflows/checks.yml
name: Required Checks

on: [pull_request]

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - run: make lint
  
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - run: make test
  
  security:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - run: make security-check
```
</MySCode.Cell>

<VSCode.Cell language="markdown">
## 8. Git Workflow Decision Tree

```
Are you working solo?
├─ YES → Trunk-based development (commit directly to main)
└─ NO → Continue...

Do you need strict stability?
├─ YES → Git Flow (develop + release branches)
└─ NO → GitHub Flow (simpler)

Release frequency?
├─ Frequently (daily) → Trunk-Based Development
├─ Weekly → GitHub Flow
└─ Monthly → Git Flow
```

## Git Workflow Checklist

- [ ] Choose workflow (Git Flow, GitHub Flow, Trunk-Based)
- [ ] Set branch protection rules
- [ ] Require code reviews
- [ ] Require passing tests
- [ ] Configure automatic merge on approval
- [ ] Set up release automation
- [ ] Document branching strategy
- [ ] Train team on workflow
- [ ] Use semantic versioning
- [ ] Maintain CHANGELOG.md

**Best Practice**: Start with GitHub Flow, graduate to Trunk-Based as team matures
</MySCode.Cell>
```